# 04 · Report generation

Assemble the canonical `AuditResult` and write all three formats
(.xlsx / .docx / .json) to a temporary directory. This is exactly what the web
app and CLI do after fetching a live tenant.

In [ ]:
import sys, json
from pathlib import Path

# Repo root = parent of the notebooks/ directory.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
FIXTURES = REPO / "tests" / "fixtures"
print("repo:", REPO)
print("fixtures:", [p.name for p in FIXTURES.glob("*.json")])

In [ ]:
from m365_review.core.models import Organization, SubscribedSku, User, TenantData

org = Organization.from_graph(json.loads((FIXTURES / "organization.json").read_text())["value"][0])
skus = [SubscribedSku.from_graph(s) for s in json.loads((FIXTURES / "subscribedSkus.json").read_text())["value"]]
users = [User.from_graph(u) for u in json.loads((FIXTURES / "users.json").read_text())["value"]]

data = TenantData(
    organization=org, tenant_id=org.id, skus=skus, users=users,
    sign_in_activity_available=True,
)
print(f"{org.display_name}: {len(skus)} SKUs, {len(users)} users")

In [ ]:
import tempfile
from datetime import datetime, timezone
from m365_review.core.engine import build_result
from m365_review.core.report import write_reports

result = build_result(data, now=datetime(2026, 7, 16, tzinfo=timezone.utc))

print("Tenant           :", result.tenant_display_name)
print("Monthly savings  : $%.2f" % result.total_monthly_savings_usd)
print("Annual savings   : $%.2f" % result.total_annual_savings_usd)
print("Findings         :", len(result.findings))
print("Severity         :", result.severity_counts())

In [ ]:
out = Path(tempfile.mkdtemp(prefix="m365_nb_"))
paths = write_reports(result, output_dir=out, formats=["xlsx", "docx", "json"])
for fmt, p in paths.items():
    print(f"{fmt:5} {p.stat().st_size:>7} bytes  {p.name}")

## Inspect the JSON optimization summary

In [ ]:
from m365_review.core.report.json_writer import build_json_payload
build_json_payload(result)["optimization_summary"]